In [21]:
import tensorflow as tf

In [ ]:
print(tf.__version__)
# needs to be an earlier tensorflow version (e.g., 2.10.0) to run model

2.10.0


## Data Preprocessing (Converting to Mel Spectrograms)
Uses code from ABGQI-CNN/0_melspec_generation-py

In [ ]:
import os
import pandas as pd
import matplotlib
matplotlib.use('Agg') # No pictures displayed 
import pylab
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
import joblib
import math
from tqdm import tqdm

In [ ]:
# retrieve files from Farm 53:
files = [os.path.join(r, f) for r, _, fs in os.walk('/Volumes/aid_elephants_interaction/Audio Data/2025/05_FR53') for f in fs if ".WAV" in f]

In [27]:
print(len(files))

933


In [ ]:
# generate predictions for first 100 files for now
files = files[:100]

100

In [ ]:
result_dir = 'results'
melspec_farm_number = 'melspecs_53'
if not (os.path.exists(os.path.join(result_dir, melspec_farm_number))):
    os.mkdir(os.path.join(result_dir, melspec_farm_number))

In [ ]:
# where the melspecs will be generated. Pathway needs to exist, next chunk will create class folders IN the folder
mel_spec_dir = os.path.join(result_dir, melspec_farm_number) # "results/melspecs_53"

roi_len = 2.000 # IMPORTANT for mfcc duration

In [ ]:
# Make a folder for each file where the respective melspecs will go
# adapted from: build a mel spec directory for each class

for i in tqdm(range(len(files))):
    # extract the name of the .wav file in order to name directories correctly
    file_name = files[i].split('/')[-1].split('.')[0]
    
    if not (os.path.exists(os.path.join(mel_spec_dir, file_name))):
        os.mkdir(os.path.join(mel_spec_dir, file_name))

print(len(os.listdir(mel_spec_dir)))

100%|██████████| 100/100 [00:00<00:00, 12358.00it/s]

100


In [ ]:
en = 0
roi_len = 2.000 # IMPORTANT for mfcc duration
n_sam = int(22050 * roi_len)

In [ ]:

# generate mel specs for each concatenated wav
for wav_nm in tqdm(files):
    rows = []
    cnt = 0
    en+=1
    y, sr = librosa.load(wav_nm, sr = 22050)
    tot_sam = int(np.shape(y)[0]/n_sam)

    # added to match time to melspec:
    roi_len_sec = n_sam / sr
    print('Total number of possible ROIs:', tot_sam)
    for n in (range(tot_sam)):
        # added to get time and save to dictionary
        start_time = n * roi_len_sec
        end_time = (n + 1) * roi_len_sec

        y_21k = y[n*n_sam:(n+1)*n_sam]

        fig = plt.figure(1, frameon=False)
        fig.set_size_inches(6, 6)
        ax = plt.Axes(fig, [0., 0., 1., 1.])
        ax.set_axis_off()
        fig.add_axes(ax)

        S = librosa.feature.melspectrogram(y=y_21k, 
                                            sr=22050, 
                                            n_mels=128,
                                            fmin = 0,                                     
                                            fmax=11025, 
                                            n_fft=728, 
                                            hop_length=32, 
                                            #win_length = None, 
                                            htk = True)

        librosa.display.specshow(librosa.power_to_db(S ** 1, ref=np.max), fmin=0, y_axis='linear')# , cmap = 'gray')
        
        file_name = wav_nm.split('/')[-1].split('.')[0]

        directory = os.path.join(mel_spec_dir, file_name, str(n)+'.png')
        fig.savefig(directory)

        rows.append({
            "file": file_name,
            "index": n,
            "start_sec": start_time,
            "end_sec": end_time
        })
        
        fig.clear()
        ax.cla()
        plt.clf()
        plt.close('all')
        #cnt +=1
        #print('Number of ROIs converted here:', cnt)


    df = pd.DataFrame(rows)
    csv_path = os.path.join(mel_spec_dir, file_name, 'metadata.csv')
    df.to_csv(csv_path, index=False)

  0%|          | 0/30 [00:00<?, ?it/s]

Total number of possible ROIs: 1797


  3%|▎         | 1/30 [04:03<1:57:48, 243.72s/it]

Total number of possible ROIs: 1797


  7%|▋         | 2/30 [07:56<1:50:39, 237.13s/it]

...

 93%|█████████▎| 28/30 [1:27:51<07:22, 221.04s/it]

Total number of possible ROIs: 1797


 97%|█████████▋| 29/30 [1:31:34<03:41, 221.89s/it]

Total number of possible ROIs: 1797


100%|██████████| 30/30 [1:35:16<00:00, 190.55s/it]


## Inference
Uses code from ABGQI-CNN/2_cnn_inferency-py

In [ ]:
# model_path = "CQuinn8/CQuinn8-ABGQI-CNN-93420d1/ABGQI-CNN"
model_path = 'ABGQI-CNN/ABGQI-CNN'
model = tf.keras.models.load_model(model_path)

In [36]:
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_1.40_224 (Funct  (None, 7, 7, 1792)       4363712   
 ional)                                                          
                                                                 
 global_average_pooling2d (G  (None, 1792)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 5)                 8965      
                                                                 
Total params: 4,372,677
Trainable params: 8,965
Non-trainable params: 4,363,712
_________________________________________________________________


In [37]:
# dimensions of our images    
img_width, img_height, img_depth = 224, 224, 3 # 224 pixels x 224 pixels x 3 bands (RGB)
# number of images to predict at once 
batch_size = 50

In [ ]:
# function to read in a file path
def process_path(file_path, IMG_HEIGHT, IMG_WIDTH):
    # load the raw data from the file as a string
    img = tf.io.read_file(file_path)
    img = decode_img(img, IMG_HEIGHT, IMG_WIDTH)
    return img #, label

# function that interpretes image after being provided a path in process_path
def decode_img(img, IMG_HEIGHT, IMG_WIDTH):
  # convert the compressed string to a 3D uint8 tensor
  img = tf.image.decode_jpeg(img, channels=3)
  # Use `convert_image_dtype` to convert to floats in the [0,1] range.
  img = tf.image.convert_image_dtype(img, tf.float32)
  # resize the image to the desired size.
  return tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])

In [39]:
import os, glob
import pandas as pd

## Predict
Uses code from ABGQI-CNN/2_cnn_inference-py

In [ ]:
#output directory
out_dir = 'results'

# data dir for images to predict class
data_path = mel_spec_dir

# number of cores
cpus = 2

# print(checkpoint_path)
print(out_dir)

png_dirs = []
for filename in os.listdir(data_path):
    if os.path.isdir(os.path.join(data_path, filename)):
        png_dirs.append(filename)
# if len(png_dirs) == 0: 
#     png_dirs.append(data_path)

print(len(png_dirs))

results/melspecs_53
100


In [53]:
#for i, f in (enumerate(tqdm(png_dirs))):
f = '20250212_180000'
temp_dir = os.path.join(data_path, f)
print('temp_dir', temp_dir)
pngs = glob.glob(os.path.join(temp_dir, "*.png"))

pngs = sorted(
    pngs,
    key=lambda x: int(os.path.splitext(os.path.basename(x))[0])
)

temp_class = f
if len(pngs) != 0:
    print("Number of pngs", len(pngs))


# list to hold each dir of predictions (either all pngs in class of mfccs in wav)
sigmoid_pred_lst = []
pred_lst = []

# iterate through each melspec in file directory
for j in tqdm(range(len(pngs))):
    temp_png = pngs[j] # png
    # print(temp_png)

    img_ = process_path(temp_png, IMG_HEIGHT = img_height, IMG_WIDTH = img_width)
    img_ = tf.reshape(img_, shape=(1, img_height, img_width, img_depth))

    # get predictions
    pred = model.predict(img_, verbose=0, steps=1, callbacks=None, max_queue_size=10,
                            workers=cpus, use_multiprocessing=False)
    
    # png_num = temp_png.split('/')[-1].split('.')[0]
    pred_lst.append(pred) # append original prediction
    sigmoid_pred = tf.math.sigmoid(pred).numpy()
    sigmoid_pred_lst.append(sigmoid_pred)

# save label preds
flat_sigmoid = [item for sublist in sigmoid_pred_lst for item in sublist]
df_sigmoid = pd.DataFrame(flat_sigmoid)
df_sigmoid.columns = ["Anthrophony", "Biophony", "Geophony", "Other", "Interference"]

flat_pred = [item for sublist in pred_lst for item in sublist]
df_pred = pd.DataFrame(flat_pred)
df_pred.columns = ["Anthrophony", "Biophony", "Geophony", "Other", "Interference"]

png_ids = [
    int(os.path.splitext(os.path.basename(p))[0])
    for p in pngs
]

df_sigmoid["png_id"] = png_ids
df_pred["png_id"] = png_ids



# write directory csv 
df_pred.to_csv(os.path.join(out_dir, f, 'pred.csv'))
df_sigmoid.to_csv(os.path.join(out_dir, f, 'sigmoid.csv'))

print('Saved this pkl at:',os.path.join(out_dir, 'predictions'), "\nLength was:",len(sigmoid_pred_lst))

temp_dir results/melspecs_53/20250212_180000
Number of pngs 1797


  0%|          | 0/1797 [00:00<?, ?it/s]

100%|██████████| 1797/1797 [01:11<00:00, 25.28it/s]


Saved this pkl at: results/melspecs_53/predictions 
Length was: 1797


## Evaluating Outputs on Sri Lanka Audio Data

In [ ]:
file_names = [name for name in os.listdir(mel_spec_dir) if os.path.isdir(os.path.join(mel_spec_dir, name))]

df = pd.DataFrame(columns = ['file', 'start_sec', 'end_sec', 'Anthrophony', 'Biophony', 'Geophony', 'Other', 'Interference', 'png_id'])

for name in file_names:
    meta_df = pd.read_csv(os.path.join(mel_spec_dir, name, 'metadata.csv'))
    sigmoid_df = pd.read_csv(os.path.join(mel_spec_dir, name, 'sigmoid.csv'))
    joined_df = meta_df.join(sigmoid_df)
    joined_df = joined_df[['file', 'start_sec', 'end_sec', 'Anthrophony', 'Biophony', 'Geophony', 'Other', 'Interference', 'png_id']]

    df = pd.concat([df, joined_df], axis=0)

print(df.shape)
df.head()


/var/folders/xs/50kmp8g92x99q2dxvv031c980000gn/T/ipykernel_70332/3835927816.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, joined_df], axis=0)


(171591, 9)


,file,start_sec,end_sec,Anthrophony,Biophony,Geophony,Other,Interference,png_id
0,20250212_010000,0.0,2.0,0.125827,0.801368,0.002414,0.001985,0.032856,0
1,20250212_010000,2.0,4.0,0.087447,0.853661,0.013001,0.011534,0.005409,1
2,20250212_010000,4.0,6.0,0.372696,0.470370,0.042632,0.056884,0.037892,2
3,20250212_010000,6.0,8.0,0.247461,0.759065,0.019825,0.020429,0.019022,3
4,20250212_010000,8.0,10.0,0.292027,0.264069,0.058576,0.006323,0.004152,4


In [55]:
df.describe()

,start_sec,end_sec,Anthrophony,Biophony,Geophony,Other,Interference
count,171591.000000,171591.000000,171591.000000,171591.000000,171591.000000,171591.000000,171591.000000
mean,1787.877616,1789.877616,0.299647,0.477939,0.060612,0.010677,0.026212
std,1041.091688,1041.091688,0.261992,0.305428,0.089211,0.012285,0.031364
min,0.000000,2.000000,0.000124,0.000077,0.000038,0.000044,0.000034
25%,884.000000,886.000000,0.085340,0.191177,0.011001,0.003846,0.008645
50%,1786.000000,1788.000000,0.212156,0.478890,0.026721,0.007130,0.017155
75%,2690.000000,2692.000000,0.459524,0.756816,0.069535,0.013070,0.032659
max,3592.000000,3594.000000,0.999614,0.999785,0.967217,0.557463,0.949311


In [ ]:
df.to_csv(os.path.join(mel_spec_dir, 'all_predictions.csv'))